<a href="https://colab.research.google.com/github/denisejroth/bags-vectors-transformers/blob/main/bonus/notebooks/3_web_scraping_and_apis_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Bonus: Web Scraping and APIs

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

Much of the text social scientists study lives on the web: news articles, party websites,
parliamentary records, scholarly abstracts, encyclopedia pages. This notebook covers the two ways to
collect it with code:

- **APIs**: the "front door". The website offers data in a structured format, with rules on how much
  you may take. Always the first choice when one exists.
- **Scraping**: downloading web pages and picking the information out of the HTML yourself. The
  fallback when there is no API.

Both raise legal and ethical questions, so we start there.

In this notebook you will:

- go through a checklist before collecting anything
- learn just enough HTTP to make polite, robust requests
- read a site's `robots.txt` with Python
- scrape a site built for practice, page by page, with a cache
- use two real APIs: **OpenAlex** (scholarly works) and **Wikipedia**
- see what to do with sites that need JavaScript, and with social media platforms

**Contents**

1. Before you collect anything
2. HTTP in five minutes
3. Reading robots.txt
4. Parsing HTML with BeautifulSoup
5. Many pages: pagination and caching
6. APIs: OpenAlex
7. APIs: Wikipedia
8. Harder cases and good practice

No GPU needed.

There are 5 exercises along the way, marked ✏️. A separate solutions notebook has worked answers, but try them yourself first.

## 0. Setup

`requests` downloads pages, `BeautifulSoup` (from the `beautifulsoup4` package) reads HTML. Both are
already installed on Colab.

In [ ]:
!pip install -q requests beautifulsoup4

import hashlib
import json
import os
import time
from pathlib import Path
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser

import matplotlib.pyplot as plt
import pandas as pd
import requests
from bs4 import BeautifulSoup

pd.set_option("display.max_colwidth", 100)

## 1. Before you collect anything

Work through these questions *before* writing a scraper. They decide whether you should scrape at
all, and how.

| Question | Why it matters |
|---|---|
| **Does a ready-made dataset exist?** | Someone may already have collected and cleaned it (data archives, replication files, Kaggle, Hugging Face). |
| **Is there an API?** | Structured data, clear rules, far less work, and far less likely to break. |
| **What do the terms of service say?** | Many sites forbid automated collection. Breaking the terms may not be illegal, but it can be a contract issue, and your university will want to know. |
| **What does `robots.txt` allow?** | The site's machine-readable wishes for bots (Section 3). Not law, but ignoring it is hard to defend. |
| **Is there personal data?** | Names, usernames, photos and opinions are personal data under the GDPR, even when public. You need a legal basis, data minimization, and a data management plan. |
| **Copyright?** | Copying texts for analysis is usually covered for research. In the EU, research organizations have a text-and-data-mining exception (Art. 3 of the 2019 Copyright Directive), but *republishing* the texts generally is not. Share derived data (counts, IDs), not full texts. |
| **Will you burden the server?** | Small sites can be overwhelmed by careless scripts. Go slowly. |

> This is not legal advice, and the rules differ by country and platform. For a real project, talk to
> your faculty's data steward or privacy officer early: they see these questions all the time.

## 2. HTTP in five minutes

Every time your browser opens a page, it sends a **request** to a server and gets a **response**
back. The response has a **status code**, some **headers** (metadata), and a **body** (the HTML, or
JSON for an API).

| Status | Meaning |
|---|---|
| `200` | OK |
| `301`, `302` | moved; `requests` follows these automatically |
| `403` | forbidden: you're not allowed (sometimes: the site blocks bots) |
| `404` | not found |
| `429` | too many requests: slow down |
| `500`-`504` | server error: try again later |

We practice on [quotes.toscrape.com](https://quotes.toscrape.com), a site made for learning to
scrape.

In [ ]:
response = requests.get("https://quotes.toscrape.com/", timeout=20)

print("Status:      ", response.status_code)
print("Content type:", response.headers.get("Content-Type"))
print("Size:        ", len(response.text), "characters")
print()
print(response.text[:600])

### Being polite

A well-behaved script:

- **identifies itself** with a `User-Agent` header that says who you are and how to reach you
  (Wikipedia, for one, requires this);
- uses a **timeout**, so it doesn't hang forever;
- **waits** between requests;
- **retries** a few times after a `429` or a server error, waiting longer each time ("exponential
  backoff"), and respects the server's `Retry-After` header if it sends one.

A `Session` keeps the same headers (and connection) for all requests. Put your own e-mail address
in the `User-Agent` below.

In [ ]:
session = requests.Session()
session.headers.update({
    "User-Agent": "BVT-course/1.0 (computational text analysis course; contact: your.name@wur.nl)"
})

MIN_DELAY = 1.0          # seconds between requests
_last_request = 0.0


def polite_get(url, params=None, max_tries=4):
    # GET a URL: wait between requests, retry on 429 and server errors, raise on other errors
    global _last_request
    for attempt in range(1, max_tries + 1):
        wait = MIN_DELAY - (time.time() - _last_request)
        if wait > 0:
            time.sleep(wait)
        _last_request = time.time()
        try:
            r = session.get(url, params=params, timeout=20)
        except requests.RequestException:                 # network trouble: wait and retry
            if attempt == max_tries:
                raise
            time.sleep(2 ** attempt)
            continue
        if r.status_code in (429, 500, 502, 503, 504) and attempt < max_tries:
            retry_after = r.headers.get("Retry-After", "")
            time.sleep(float(retry_after) if retry_after.isdigit() else 2 ** attempt)
            continue
        r.raise_for_status()                               # 403, 404, ...: stop with an error
        return r


r = polite_get("https://quotes.toscrape.com/")
print(r.status_code, "| sent as:", r.request.headers["User-Agent"])

## 3. Reading robots.txt

Most sites publish a file at `/robots.txt` that tells bots which parts they may visit. Each block
starts with the `User-agent` it applies to (`*` means everyone), followed by `Disallow` rules for
paths, and sometimes a `Crawl-delay` in seconds. Many news sites now also disallow AI crawlers such as
`GPTBot` by name.

Python's built-in `RobotFileParser` reads these rules for you. Here's an example file for a
made-up news site:

In [ ]:
EXAMPLE_ROBOTS = '''
User-agent: *
Disallow: /search
Disallow: /account/
Crawl-delay: 5

User-agent: GPTBot
Disallow: /
'''

rp = RobotFileParser()
rp.parse(EXAMPLE_ROBOTS.splitlines())

for url in ["https://news.example.org/politics/2026/budget-deal",
            "https://news.example.org/search?q=nitrogen",
            "https://news.example.org/account/settings"]:
    print(f"{rp.can_fetch('BVT-course', url)!s:<6} {url}")
print("Crawl delay asked for:", rp.crawl_delay("BVT-course"), "seconds")

Now the real thing. When a site has no `robots.txt` at all (status 404), nothing is disallowed; if
the file itself is forbidden (401 or 403), the convention is to treat the whole site as off limits.

In [ ]:
def robots_for(site):
    # Download and parse a site's robots.txt
    rp = RobotFileParser()
    r = session.get(urljoin(site, "/robots.txt"), timeout=20)
    if r.status_code == 200:
        rp.parse(r.text.splitlines())
    elif r.status_code in (401, 403):
        rp.disallow_all = True
    else:
        rp.parse([])                                      # no robots.txt: nothing disallowed
    return rp


BASE = "https://quotes.toscrape.com/"
robots = robots_for(BASE)
print("robots.txt status:", session.get(urljoin(BASE, "/robots.txt"), timeout=20).status_code)
print("May we fetch the first page?", robots.can_fetch(session.headers["User-Agent"], BASE))

> **✏️ Exercise 1**
>
> Using `rp` (the example news site above):
>
> 1. May a generic bot fetch `https://news.example.org/searchresults/archive`? Why (look closely at
>    the rule)?
> 2. May `GPTBot` fetch the homepage? Now try the full user-agent string that crawler actually
>    sends, `Mozilla/5.0 (compatible; GPTBot/1.1)`. What happens, and what does it tell you about
>    which name to pass to `can_fetch`?
> 3. How long should `polite_get` wait between requests to this site? Change `MIN_DELAY` to match
>    if needed (and set it back to `1.0` afterwards).

In [ ]:
# Your code here

## 4. Parsing HTML with BeautifulSoup

HTML is a tree of **tags**: `<div class="quote">` contains a `<span class="text">`, and so on. To find
out where the information you want lives, open the page in your browser, right-click on it and
choose **Inspect**. You'll see the tag, its `class`, and what it sits inside.

BeautifulSoup turns the HTML into a tree you can search with **CSS selectors**:

| Selector | Finds |
|---|---|
| `div.quote` | `<div>` tags with class `quote` |
| `span.text` | `<span>` tags with class `text` |
| `div.tags a.tag` | `<a class="tag">` *inside* a `<div class="tags">` |
| `li.next a` | the link inside `<li class="next">` |
| `a[href^='/author/']` | links whose `href` starts with `/author/` |

Let's look at the first quote on the page:

In [ ]:
soup = BeautifulSoup(r.text, "html.parser")
first = soup.select_one("div.quote")
print(first.prettify()[:1400])

In [ ]:
print("Text:  ", first.select_one("span.text").get_text(strip=True))
print("Author:", first.select_one("small.author").get_text(strip=True))
print("Tags:  ", [a.get_text(strip=True) for a in first.select("div.tags a.tag")])
print("Link:  ", first.select_one("a[href^='/author/']")["href"])

`.get_text(strip=True)` gives the text without surrounding whitespace, and `["href"]` reads an
attribute. Links are often *relative* (`/author/...`); `urljoin` turns them into full URLs.

Now we wrap this in a function that parses a whole page, and also returns the link to the next page
(or `None` on the last page):

In [ ]:
def parse_quotes(html, page_url):
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for q in soup.select("div.quote"):
        author_link = q.select_one("a[href^='/author/']")
        rows.append({
            "text": q.select_one("span.text").get_text(strip=True).strip("“”\""),
            "author": q.select_one("small.author").get_text(strip=True),
            "tags": [a.get_text(strip=True) for a in q.select("div.tags a.tag")],
            "author_url": urljoin(page_url, author_link["href"]) if author_link else None,
        })
    next_link = soup.select_one("li.next a")
    next_url = urljoin(page_url, next_link["href"]) if next_link else None
    return rows, next_url


rows, next_url = parse_quotes(r.text, BASE)
print(len(rows), "quotes on this page; next page:", next_url)
pd.DataFrame(rows).head()

Scrapers are **fragile**: when the site changes its HTML, selectors silently stop matching. Note how
`parse_quotes` would crash with an unhelpful error if a quote had no `small.author`. For a long-running
project, check the number of rows per page and look at a sample now and then.

## 5. Many pages: pagination and caching

To get everything, we follow the "Next" link until there isn't one. Two habits make this much better:

1. **Separate downloading from parsing.** Save the raw HTML to disk first, then parse the saved
   files. If your parser has a bug (it will), you fix it and re-parse, without downloading again.
2. **Cache.** Before downloading a URL, check whether you already have it. Re-running the notebook
   then costs the server nothing.

The cache also documents your data collection: we log each URL with the time it was downloaded.

In [ ]:
CACHE = Path("cache")
CACHE.mkdir(exist_ok=True)


def fetch_cached(url):
    # Return the HTML for url, from the cache if we have it, otherwise download and save it
    path = CACHE / (hashlib.sha1(url.encode()).hexdigest()[:16] + ".html")
    if path.exists():
        return path.read_text(encoding="utf-8")
    html = polite_get(url).text
    path.write_text(html, encoding="utf-8")
    with open(CACHE / "index.tsv", "a", encoding="utf-8") as log:
        log.write(f"{path.name}\t{url}\t{time.strftime('%Y-%m-%d %H:%M:%S')}\n")
    return html


def scrape_quotes(start_url, max_pages=20):
    rows, url, pages = [], start_url, 0
    while url and pages < max_pages:
        if not robots.can_fetch(session.headers["User-Agent"], url):
            print("robots.txt disallows", url); break
        page_rows, next_url = parse_quotes(fetch_cached(url), url)
        for row in page_rows:
            row["page_url"] = url
        rows += page_rows
        url, pages = next_url, pages + 1
    return pd.DataFrame(rows)


start = time.time()
quotes = scrape_quotes(BASE)
print(f"{len(quotes)} quotes from {quotes['page_url'].nunique()} pages in {time.time() - start:.1f} s")

In [ ]:
start = time.time()
quotes = scrape_quotes(BASE)
print(f"Again, now from the cache: {time.time() - start:.2f} s")
print(open(CACHE / "index.tsv").read()[:400])

In [ ]:
print("Most quoted authors:")
print(quotes["author"].value_counts().head(8).to_string())

print("\nMost common tags:")
print(quotes.explode("tags")["tags"].value_counts().head(8).to_string())

> **✏️ Exercise 2**
>
> Each quote links to a page about its author (the `author_url` column). Write a function
> `parse_author(html)` that returns the author's name (`h3.author-title`), birth date
> (`span.author-born-date`) and birth place (`span.author-born-location`). Use it, together with
> `fetch_cached`, to build a DataFrame for the **five most quoted authors**.

In [ ]:
# Your code here

> **✏️ Exercise 3**
>
> The tag links lead to pages that list all quotes with that tag, for example
> `https://quotes.toscrape.com/tag/inspirational/`, again spread over several pages. `scrape_quotes`
> works for these too, without any changes. Scrape all quotes tagged `inspirational`. How many
> are there, and which authors appear most often? Do they match the quotes with that tag in
> `quotes`?

In [ ]:
# Your code here

## 6. APIs: OpenAlex

An **API** (application programming interface) answers requests with structured data, usually
**JSON**: nested dictionaries and lists that Python reads directly. No HTML parsing, no guessing.

[OpenAlex](https://openalex.org) is a free, open index of hundreds of millions of scholarly works
(articles, books, datasets), with titles, abstracts, authors, institutions, venues and citations.
Useful for a literature review, or as a corpus of abstracts in itself.

### An API key

Since February 2026, OpenAlex uses **free API keys** with a daily budget: list queries cost very
little of it, full-text *searches* ten times more, and looking up a single work costs nothing. Without
a key you get a small budget, enough for this notebook but not for a project.

1. Create a free account at openalex.org and copy your key from openalex.org/settings/api.
2. In Colab, click the 🔑 icon (**Secrets**) in the left sidebar, add a secret named
   `OPENALEX_API_KEY` with your key as the value, and allow this notebook to access it.

Keeping keys in Secrets (or an environment variable) instead of in your code means you can share the
notebook without sharing your key.

In [ ]:
def get_secret(name):
    # Colab Secrets if available, otherwise an environment variable, otherwise ""
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return os.environ.get(name, "")


OPENALEX_KEY = get_secret("OPENALEX_API_KEY")
print("Using an OpenAlex API key" if OPENALEX_KEY else "No key found: using the small keyless budget")

OPENALEX = "https://api.openalex.org/works"


def years(first, last):
    # An OpenAlex filter for works published from `first` to `last` (inclusive)
    return f"from_publication_date:{first}-01-01,to_publication_date:{last}-12-31"


def openalex_params(**params):
    if OPENALEX_KEY:
        params["api_key"] = OPENALEX_KEY
    return params

A first request: works matching a search term, published 2016-2025. We ask for only five results to
look at the structure. `select` asks for only the fields we need, which keeps responses small.

In [ ]:
SELECT = "id,display_name,publication_year,cited_by_count,primary_location,abstract_inverted_index"

data = polite_get(OPENALEX, params=openalex_params(
    search="misinformation correction", filter=years(2016, 2025),
    per_page=5, select=SELECT)).json()

print("Works found:", data["meta"]["count"])
print(json.dumps(data["results"][0], indent=2)[:1500])

Two things to notice:

- `primary_location` is nested: the journal's name is in `primary_location["source"]["display_name"]`,
  and `source` can be missing (`None`), for example for preprints.
- For copyright reasons, OpenAlex doesn't store abstracts as text but as an **inverted index**:
  each word with the positions where it occurs. Putting the words back in order of position
  rebuilds the abstract.

In [ ]:
def rebuild_abstract(inverted_index):
    if not inverted_index:
        return ""
    positions = [(pos, word) for word, poss in inverted_index.items() for pos in poss]
    return " ".join(word for _, word in sorted(positions))


def flatten_work(w):
    source = (w.get("primary_location") or {}).get("source") or {}
    return {"id": w["id"], "title": w["display_name"], "year": w["publication_year"],
            "citations": w["cited_by_count"], "venue": source.get("display_name"),
            "abstract": rebuild_abstract(w.get("abstract_inverted_index"))}


example = next((w for w in data["results"] if w.get("abstract_inverted_index")), data["results"][0])
print(flatten_work(example)["abstract"][:500] or "(none of these five works has an abstract)")

### Counting without downloading: `group_by`

Want the number of works per year? Don't download them all: `group_by` makes OpenAlex count for you,
in a single request.

In [ ]:
counts = polite_get(OPENALEX, params=openalex_params(
    search="misinformation correction", filter=years(2016, 2025),
    group_by="publication_year")).json()["group_by"]

per_year = pd.DataFrame(counts)[["key", "count"]].astype({"key": int}).sort_values("key")
plt.figure(figsize=(8, 3))
plt.bar(per_year["key"], per_year["count"], color="#34B233")
plt.title('Works matching "misinformation correction"'); plt.xlabel("year"); plt.ylabel("works")
plt.show()

### Many results: cursor paging

One request returns at most 100 works (`per_page=100`). For more, use **cursor paging**: start with
`cursor="*"`, and each response gives you the cursor for the next page in `meta["next_cursor"]`,
until it's `None`.

In [ ]:
def openalex_works(max_results=200, **query):
    params = openalex_params(per_page=100, cursor="*", select=SELECT, **query)
    works = []
    while params["cursor"] and len(works) < max_results:
        data = polite_get(OPENALEX, params=params).json()
        works += data["results"]
        params["cursor"] = data["meta"].get("next_cursor")
    return pd.DataFrame([flatten_work(w) for w in works[:max_results]])


works = openalex_works(max_results=200, search="misinformation correction",
                       filter=years(2016, 2025))
print(len(works), "works;", (works["abstract"] != "").mean().round(2), "have an abstract")
works.sort_values("citations", ascending=False).head(5)[["title", "year", "citations", "venue"]]

That's a ready-made corpus of abstracts for any method from the course: TF-IDF, topic models,
embeddings. Keep in mind that a search engine decides what "matching" means, so for a systematic
review, document the exact query, filters and date.

> **✏️ Exercise 4**
>
> Pick a topic from your own research.
>
> 1. Plot the number of works per year since 2000 with `group_by`.
> 2. Download the 100 **most cited** works on the topic published since 2020 (add
>    `sort="cited_by_count:desc"`), and list the five most common venues.

In [ ]:
# Your code here

## 7. APIs: Wikipedia

Wikipedia's API gives you the plain text of any article, in any language edition. It asks every
client to send a descriptive `User-Agent` with contact details, which our `session` already does.
The parameters:

- `prop=extracts` with `explaintext=1`: the article as plain text (no HTML)
- `exintro=1`: only the introduction
- `redirects=1`: follow redirects, so "WUR" finds the right article
- `formatversion=2`: a simpler JSON layout

In [ ]:
def wiki_extract(title, lang="en", intro_only=False):
    params = {"action": "query", "prop": "extracts", "explaintext": 1, "redirects": 1,
              "titles": title, "format": "json", "formatversion": 2}
    if intro_only:
        params["exintro"] = 1
    page = polite_get(f"https://{lang}.wikipedia.org/w/api.php", params=params).json()["query"]["pages"][0]
    return None if page.get("missing") else page["extract"]


text = wiki_extract("Natura 2000", intro_only=True)
print(text[:700])

In [ ]:
print(wiki_extract("Natura 2000", lang="nl", intro_only=True)[:500])

> **✏️ Exercise 5**
>
> Combine the API with Day 1. Download the full text of these articles: `Common Agricultural
> Policy`, `Natura 2000`, `European Green Deal`, `Nitrogen cycle` and `Wageningen University and
> Research`. Then use `TfidfVectorizer(stop_words="english")` to find the eight most distinctive words
> of each article.

In [ ]:
# Your code here

## 8. Harder cases and good practice

### Pages that need JavaScript

Many modern sites send an almost empty HTML page and fill it in with JavaScript. `requests` only sees
the empty page. Two options:

1. **Find the data behind the page.** Open your browser's developer tools (**Inspect → Network**,
   filter on *Fetch/XHR*) and reload. Often the page loads its content as JSON from an internal API,
   which you can request directly. This is the cleanest route.
2. **Drive a real browser** with **Playwright** or **Selenium**: they load the page, run the
   JavaScript, and let you read the result. Slower and heavier, but works for almost anything.

### News articles

For news, the library usually beats scraping: **Nexis Uni** (LexisNexis) and similar databases have
full texts of most newspapers, and **Delpher** has Dutch newspapers back to the 17th century. To
extract the main text of a web article (without menus and ads), the `trafilatura` package is
excellent (see Bonus 1).

### Social media

Scraping social media platforms is usually against their terms, and their APIs have become much more
restricted and expensive since 2023 (the "post-API age", Freelon 2018). Current routes include the
platforms' own research programs (e.g., Meta's Content Library, TikTok's Research API) and, in the
EU, the **Digital Services Act**: Article 40 gives vetted researchers a legal route to request data
from very large platforms, with procedures set out in 2025. Expect paperwork and waiting times.

### Other useful APIs

- **Tweede Kamer Open Data**: Dutch parliamentary documents, debates and votes
- **CBS Open Data** (`cbsodata` package): Statistics Netherlands
- **The Guardian, New York Times**: article search and metadata (free keys)
- **Common Crawl**: petabytes of archived web pages, if you need the web at scale

### Good practice checklist

- Check for a dataset or an API first; read the terms and `robots.txt`
- Identify yourself, go slowly, retry with backoff
- Save the raw responses and log the URL and time of every download
- Collect only what you need, especially personal data
- Validate: count rows per page, compare routes, look at samples
- Report what you collected, when, how, and what you couldn't get

### Further reading

- Bruns, A. (2019). After the 'APIcalypse': Social media platforms and their fight against critical
  scholarly research. *Information, Communication & Society, 22*(11), 1544-1566.
- Fiesler, C., Beard, N., & Keegan, B. C. (2020). No robots, spiders, or scrapers: Legal and ethical
  regulation of data collection methods in social media terms of service. *Proceedings of ICWSM*.
- Freelon, D. (2018). Computational research in the post-API age. *Political Communication, 35*(4),
  665-668.
- Priem, J., Piwowar, H., & Orr, R. (2022). OpenAlex: A fully-open index of scholarly works, authors,
  venues, institutions, and concepts. *arXiv:2205.01833*.